# DeepSORT baseline

1. **Detect:** our own Faster R-CNN (torchvision ResNet-50-FPN, COCO weights, not trained on MOT17) finds people in every raw frame.
2. **Track:** DeepSORT (deep-sort-realtime 1.3.2, MobileNetV2 appearance network) links the boxes over time and gives each person an ID.
3. **Score:** TrackEval (HOTA, MOTA, IDF1) on MOT17-10 (validation) and MOT17-13 (test), against the unaltered `gt.txt`.

MOT17's own supplied detections are not used. Frames and ground truth come from `Exploratory data analysis.ipynb`
(run it first). Results go to `outputs/deepsort/`.

In [ ]:
import hashlib
import importlib.metadata
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings("ignore", message=".*pkg_resources.*")    # harmless warning from deep-sort-realtime
from deep_sort_realtime.deepsort_tracker import DeepSort
np.float, np.int = float, int     # TrackEval still uses these two names, removed from NumPy (they were plain aliases)
import trackeval

import mot17_data
from mot17_data import DATA_ROOT, open_frame, boxes_to_pixels

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("DEVICE:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "WARNING: no GPU, using the CPU")

# ---- Settings ----
SOURCE = "frcnn"          # "frcnn": our Faster R-CNN (the baseline) | "gt": ground-truth boxes (diagnostic only) | "model": Rowan's model
SCORE_THRESHOLD = 0.5     # keep person boxes with score >= 0.5
RERUN_DETECTOR = False    # True: ignore the saved detections and run Faster R-CNN again
DEEPSORT_SETTINGS = dict(max_age=30, n_init=3, max_cosine_distance=0.2, nn_budget=100, max_iou_distance=0.7,
                         embedder="mobilenet", embedder_gpu=DEVICE.type == "cuda", bgr=False)   # bgr=False: frames are RGB

OUT = mot17_data.PROJECT_ROOT / "outputs" / "deepsort"
for folder in ["detections", "trackers/DeepSORT/data", "repeat", "eval", "figures"]:
    (OUT / folder).mkdir(parents=True, exist_ok=True)

videos = mot17_data.read("videos").set_index("video")
videos = videos[videos["split"].isin(["val", "test"])]          # MOT17-10 and MOT17-13
frames = mot17_data.read("frames")
boxes = mot17_data.read("boxes")
gt_md5 = {v: hashlib.md5((DATA_ROOT / videos.loc[v, "folder"] / "gt" / "gt.txt").read_bytes()).hexdigest() for v in videos.index}

## 1. The detector: Faster R-CNN (COCO)
Each frame goes in at full size as an RGB 0–1 tensor; the model resizes and normalizes it inside and returns boxes
in original pixels. We keep label 1 (person) with score ≥ 0.5.

The boxes are saved in `outputs/deepsort/detections/` and reused next time if the weights, threshold and torchvision
version are the same (`RERUN_DETECTOR = True` forces a new run).

In [ ]:
detector = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, progress=False).to(DEVICE).eval()
CACHE_SETTINGS = {"weights": "COCO_V1", "score_threshold": SCORE_THRESHOLD, "torchvision": torchvision.__version__}


def detect(image):
    """Person boxes (score >= 0.5) in one full-size RGB frame: left, top, width, height (original pixels), score."""
    with torch.inference_mode():
        out = detector([TF.to_tensor(image).to(DEVICE)])[0]
    keep = (out["labels"] == 1) & (out["scores"] >= SCORE_THRESHOLD)      # label 1 = person
    x1, y1, x2, y2 = out["boxes"][keep].cpu().numpy().T
    return pd.DataFrame({"left": x1, "top": y1, "width": x2 - x1, "height": y2 - y1,
                         "score": out["scores"][keep].cpu().numpy()}).round(2)


def get_detections(video):
    """All boxes DeepSORT gets for a video, and the detector's time in seconds."""
    if SOURCE == "model":
        raise NotImplementedError("Rowan's model doesn't produce boxes yet")
    if SOURCE == "gt":     # diagnostic only: the ground-truth boxes as a "perfect detector" (no IDs given)
        b = boxes[boxes["video"] == video]
        det = boxes_to_pixels(b, videos.loc[video, "width"], videos.loc[video, "height"])
        return det.assign(frame=b["frame"].values, score=1.0), 0.0
    csv_file, info_file = OUT / "detections" / f"{video}.csv", OUT / "detections" / f"{video}.json"
    if not RERUN_DETECTOR and info_file.exists() and json.loads(info_file.read_text())["settings"] == CACHE_SETTINGS:
        return pd.read_csv(csv_file), json.loads(info_file.read_text())["seconds"]
    tables, seconds = [], 0.0
    for row in frames[frames["video"] == video].itertuples():
        image = open_frame(row.image_path)
        start = time.time()
        tables.append(detect(image).assign(frame=row.frame))
        seconds += time.time() - start
    det = pd.concat(tables, ignore_index=True)
    det.to_csv(csv_file, index=False)
    info_file.write_text(json.dumps({"settings": CACHE_SETTINGS, "seconds": seconds}))
    return det, seconds

## 2. Tracking with DeepSORT
One fresh tracker per video. For each frame in order: its boxes → `tracker.update_tracks(boxes, frame=image)` →
write the confirmed tracks that were matched in this frame or the one before (MOTChallenge format).

In [ ]:
def track(video, det, out_file):
    """Run DeepSORT over one video's boxes and write the tracks. Returns DeepSORT's time in seconds."""
    tracker = DeepSort(**DEEPSORT_SETTINGS)
    det_by_frame = dict(list(det.groupby("frame")))
    lines, seconds = [], 0.0
    for row in frames[frames["video"] == video].itertuples():
        image = np.asarray(open_frame(row.image_path))
        d = det_by_frame.get(row.frame, det.iloc[:0])
        start = time.time()
        tracks = tracker.update_tracks([([r.left, r.top, r.width, r.height], r.score, "person") for r in d.itertuples()],
                                       frame=image)
        seconds += time.time() - start
        for t in tracks:
            if t.is_confirmed() and t.time_since_update <= 1:
                left, top, width, height = t.to_ltwh()
                lines.append(f"{row.frame},{t.track_id},{left:.2f},{top:.2f},{width:.2f},{height:.2f},1,-1,-1,-1")
    out_file.write_text("\n".join(lines) + "\n")
    return seconds


detections, speed = {}, []
for video in videos.index:
    det, det_seconds = get_detections(video)
    track_seconds = track(video, det, OUT / "trackers" / "DeepSORT" / "data" / f"{video}-FRCNN.txt")
    detections[video] = det
    n = videos.loc[video, "length"]
    speed.append({"video": video, "detections": len(det), "detector_fps": n / det_seconds if det_seconds else np.nan,
                  "tracker_fps": n / track_seconds, "end_to_end_fps": n / (det_seconds + track_seconds)})
    print(speed[-1])

## 3. Scores (TrackEval)
MOTChallenge evaluation: only pedestrians with `use = 1` count, and boxes on distractors (static people, reflections, …) aren't penalised.

In [ ]:
def score(trackers_folder, tracker_names, video_names):
    """HOTA, MOTA, IDF1, ID switches, false positives and false negatives per tracker and video."""
    eval_config = {**trackeval.Evaluator.get_default_eval_config(), "USE_PARALLEL": False, "PRINT_RESULTS": False,
                   "PRINT_CONFIG": False, "TIME_PROGRESS": False, "PLOT_CURVES": False}
    dataset_config = {**trackeval.datasets.MotChallenge2DBox.get_default_dataset_config(),
                      "GT_FOLDER": str(DATA_ROOT / "train"), "TRACKERS_FOLDER": str(trackers_folder),
                      "TRACKERS_TO_EVAL": tracker_names, "OUTPUT_FOLDER": str(trackers_folder.parent / "eval"),
                      "BENCHMARK": "MOT17", "SPLIT_TO_EVAL": "train", "SKIP_SPLIT_FOL": True, "DO_PREPROC": True,
                      "SEQ_INFO": {f"{v}-FRCNN": None for v in video_names}, "PRINT_CONFIG": False}
    metrics = [m({"THRESHOLD": 0.5, "PRINT_CONFIG": False}) for m in
               [trackeval.metrics.HOTA, trackeval.metrics.CLEAR, trackeval.metrics.Identity]]
    results, _ = trackeval.Evaluator(eval_config).evaluate([trackeval.datasets.MotChallenge2DBox(dataset_config)], metrics)
    rows = []
    for name in tracker_names:
        for v in video_names:
            r = results["MotChallenge2DBox"][name][f"{v}-FRCNN"]["pedestrian"]
            rows.append({"tracker": name, "video": v, "HOTA": 100 * r["HOTA"]["HOTA"].mean(),
                         "MOTA": 100 * r["CLEAR"]["MOTA"], "IDF1": 100 * r["Identity"]["IDF1"],
                         "ID_switches": int(r["CLEAR"]["IDSW"]), "false_positives": int(r["CLEAR"]["CLR_FP"]),
                         "false_negatives": int(r["CLEAR"]["CLR_FN"])})
    return pd.DataFrame(rows)


results = score(OUT / "trackers", ["DeepSORT"], list(videos.index)).drop(columns="tracker").merge(pd.DataFrame(speed), on="video")
results.to_csv(OUT / "results.csv", index=False)
display(results.round(2))

## 4. Detector recall on MOT17-13
The share of ground-truth people the detector finds: each frame's ground-truth boxes are matched one-to-one to
detections, and a match counts if the boxes overlap with IoU ≥ 0.5.

In [ ]:
def iou(a, b):
    """IoU of every box in a with every box in b ([left, top, width, height] rows)."""
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 0] + a[:, None, 2], b[None, :, 0] + b[None, :, 2])
    y2 = np.minimum(a[:, None, 1] + a[:, None, 3], b[None, :, 1] + b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / (a[:, None, 2] * a[:, None, 3] + b[None, :, 2] * b[None, :, 3] - inter)


BOX = ["left", "top", "width", "height"]
gt13 = boxes[boxes["video"] == "MOT17-13"]
gt13_px = boxes_to_pixels(gt13, videos.loc["MOT17-13", "width"], videos.loc["MOT17-13", "height"]).assign(frame=gt13["frame"].values)
det13 = detections["MOT17-13"]
matched = 0
for frame, g in gt13_px.groupby("frame"):
    d = det13[det13["frame"] == frame]
    if len(d):
        overlap = iou(g[BOX].to_numpy(), d[BOX].to_numpy())
        r, c = linear_sum_assignment(-overlap)
        matched += (overlap[r, c] >= 0.5).sum()
recall = matched / len(gt13_px)
print(f"Detector recall on MOT17-13: {matched:,} of {len(gt13_px):,} ground-truth boxes found = {100 * recall:.1f}%")

## 5. One frame, start to finish (MOT17-13, frame 100)
1. **Before:** the raw frame with every raw `gt.txt` row (dashed grey = removed by cleaning).
2. **Processed input:** what Faster R-CNN's network receives after its own resize and normalization (image only, no boxes).
3. **Detector output:** person boxes with their scores.
4. **After the whole model:** DeepSORT's tracks with IDs; dashed blue = cleaned ground truth.

In [ ]:
SAMPLE_VIDEO, SAMPLE_FRAME = "MOT17-13", 100
image = open_frame(frames.loc[(frames["video"] == SAMPLE_VIDEO) & (frames["frame"] == SAMPLE_FRAME), "image_path"].iloc[0])
raw = pd.read_csv(DATA_ROOT / videos.loc[SAMPLE_VIDEO, "folder"] / "gt" / "gt.txt", header=None,
                  names=["frame", "id", "left", "top", "width", "height", "use", "class", "visibility"])
raw = raw[raw["frame"] == SAMPLE_FRAME]
kept = gt13_px[gt13_px["frame"] == SAMPLE_FRAME]
det = det13[det13["frame"] == SAMPLE_FRAME]
tracks = pd.read_csv(OUT / "trackers" / "DeepSORT" / "data" / f"{SAMPLE_VIDEO}-FRCNN.txt", header=None,
                     names=["frame", "id", *BOX, "conf", "x", "y", "z"])
tracks = tracks[tracks["frame"] == SAMPLE_FRAME]

with torch.inference_mode():       # the network's real input: the model's own resize + normalize (+ padding)
    images, _ = detector.transform([TF.to_tensor(image).to(DEVICE)])
mean = torch.tensor(detector.transform.image_mean)[:, None, None]
std = torch.tensor(detector.transform.image_std)[:, None, None]
network_input = (images.tensors[0].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)   # normalization undone to display


def draw(ax, rows, colour, labels=None, dashed=False):
    for k, r in enumerate(rows.itertuples()):
        ax.add_patch(Rectangle((r.left, r.top), r.width, r.height, fill=False, linewidth=2,
                               edgecolor=colour(r) if callable(colour) else colour, linestyle="--" if dashed else "-"))
        if labels is not None:
            ax.text(r.left, r.top - 4, labels[k], color=colour(r) if callable(colour) else colour, fontsize=10, fontweight="bold")


is_kept = (raw["class"] == 1) & (raw["use"] == 1)
panels = [
    (image, f"1. Before: raw frame + all {len(raw)} gt.txt rows (blue = kept, dashed grey = removed by cleaning)",
     lambda ax: (draw(ax, raw[is_kept], "#2a78d6"), draw(ax, raw[~is_kept], "#b5b4ad", dashed=True))),
    (network_input, f"2. Processed input: what the network receives, {tuple(images.tensors.shape[2:])} (image only, no boxes)",
     lambda ax: None),
    (image, f"3. Faster R-CNN output: {len(det)} person boxes with score >= {SCORE_THRESHOLD}",
     lambda ax: draw(ax, det, "#eb6834", labels=[f"{s:.2f}" for s in det["score"]])),
    (image, f"4. After the whole model: {len(tracks)} DeepSORT tracks with IDs (dashed blue = cleaned ground truth)",
     lambda ax: (draw(ax, kept, "#2a78d6", dashed=True),
                 draw(ax, tracks, lambda r: plt.cm.tab20(r.id % 20), labels=[str(i) for i in tracks["id"]]))),
]
for k, (picture, title, add_boxes) in enumerate(panels, start=1):
    fig, ax = plt.subplots(figsize=(16, 9.4))
    ax.imshow(picture)
    add_boxes(ax)
    ax.set_title(title, fontsize=13)
    ax.axis("off")
    fig.savefig(OUT / "figures" / f"sample_step{k}.png", dpi=90, bbox_inches="tight")
    plt.show()

## 6. Checks

In [ ]:
all_det = pd.concat(detections.values())
inside = ((all_det["left"] >= 0) & (all_det["top"] >= 0) & (all_det["left"] + all_det["width"] <= 1920)
          & (all_det["top"] + all_det["height"] <= 1080)).all()
# The detector works internally on a 1333 x 750 image; boxes reaching past that prove they're in original pixels.
print("Boxes in original pixels:", inside and (all_det["left"] + all_det["width"]).max() > 1333)
print(f"Detector recall on MOT17-13: {100 * recall:.1f}%")
for r in results.itertuples():
    print(f"MOTA above 0 on {r.video}: {r.MOTA > 0} ({r.MOTA:.1f}%)")

track("MOT17-13", detections["MOT17-13"], OUT / "repeat" / "MOT17-13-FRCNN.txt")
same = (OUT / "repeat" / "MOT17-13-FRCNN.txt").read_bytes() == (OUT / "trackers" / "DeepSORT" / "data" / "MOT17-13-FRCNN.txt").read_bytes()
print("Second tracking run from the saved detections gives an identical file:", same)
print("gt.txt unchanged:", all(hashlib.md5((DATA_ROOT / videos.loc[v, "folder"] / "gt" / "gt.txt").read_bytes()).hexdigest()
                               == gt_md5[v] for v in videos.index))

In [ ]:
# 4 frames of MOT17-13 with track IDs, spread over the life of the longest-lived track.
all_tracks = pd.read_csv(OUT / "trackers" / "DeepSORT" / "data" / "MOT17-13-FRCNN.txt", header=None,
                         names=["frame", "id", *BOX, "conf", "x", "y", "z"])
longest = all_tracks.groupby("id")["frame"].agg(lambda f: f.max() - f.min()).idxmax()
life = all_tracks.loc[all_tracks["id"] == longest, "frame"].to_numpy()
fig, axes = plt.subplots(2, 2, figsize=(18, 10.5))
for ax, frame in zip(axes.flat, life[np.linspace(0, len(life) - 1, 4).astype(int)]):
    ax.imshow(open_frame(frames.loc[(frames["video"] == "MOT17-13") & (frames["frame"] == frame), "image_path"].iloc[0]))
    t = all_tracks[all_tracks["frame"] == frame]
    draw(ax, t, lambda r: plt.cm.tab20(r.id % 20), labels=[str(i) for i in t["id"]])
    ax.set_title(f"MOT17-13 frame {frame} (ID {longest} is the longest-lived track)")
    ax.axis("off")
plt.tight_layout()
fig.savefig(OUT / "figures" / "tracks_MOT17-13.png", dpi=90)
plt.show()

## 7. Report

In [ ]:
packages = {p: importlib.metadata.version(p) for p in
            ["torch", "torchvision", "deep-sort-realtime", "trackeval", "numpy"]}
report = f"""# DeepSORT baseline report

**Models:** Faster R-CNN ResNet-50-FPN (torchvision, COCO_V1 weights) as the detector; MobileNetV2 (ImageNet weights,
half precision on the GPU, the package default) as DeepSORT's appearance network; DeepSORT as the tracker. None was trained on MOT17.

**Device:** {torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"}. **Packages:** {packages}.

**Settings:** box source `{SOURCE}`, person score >= {SCORE_THRESHOLD}, DeepSORT {DEEPSORT_SETTINGS}.

## Results (HOTA, MOTA, IDF1 in %; speeds in frames per second)

```
{results.round(2).to_string(index=False)}
```

Detector recall on MOT17-13 (IoU >= 0.5): {100 * recall:.1f}%.

## Limitations
- The detector is COCO-pretrained, not trained on MOT17, so it misses people and boxes some the ground truth ignores.
- Speeds are for one GPU and include reading the JPEGs; the appearance network is MobileNetV2 (ImageNet), not a person re-identification network.
"""
(OUT / "deepsort_report.md").write_text(report, encoding="utf-8")
print(report)